# GNNExplainer on Cora Dataset (Node Classification)

This tutorial demonstrates GNNExplainer for **node classification** on the
**Cora** citation network dataset.

Workflow:
1. Load the Cora dataset (single citation graph with 2708 nodes, 7 classes).
2. Train a GCN for semi-supervised node classification.
3. Implement `GNNInterface` for node-level explanations.
4. Run GNNExplainer to explain the prediction for a specific node.
5. Visualize the subgraph explanation showing which neighbors and edges
   are most important for the prediction.

Based on: Ying et al., GNNExplainer: Generating Explanations for Graph Neural Networks (NeurIPS 2019).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import time
from sklearn.model_selection import train_test_split
from torch_geometric.data import Batch

from kgcnn_torch.models.gcn import GCNModel
from kgcnn_torch.models.gnnexplain import GNNExplainer, GNNInterface
from kgcnn_torch.utils.devices import get_device

device = get_device("auto")
print(f"Using device: {device}")

## 1. Load Cora Dataset

Cora is a **single-graph** citation network with:
- 2708 nodes (publications)
- 5429 edges (citations)
- 1432-dim binary word vectors as node features (one column is node ID, which we remove)
- 7 classes: Case-Based, Genetic Algorithms, Neural Networks, Probabilistic Methods,
  Reinforcement Learning, Rule Learning, Theory

In [ ]:
from kgcnn_torch.data.datasets.CoraLuDataset import CoraLuDataset

cora_dataset = CoraLuDataset()
print(f"Cora dataset: {len(cora_dataset)} graph(s)")

# Single graph
cora_graph = cora_dataset[0]
print(f"Graph: {cora_graph}")
print(f"Nodes: {cora_graph.num_nodes}")
print(f"Edges: {cora_graph.num_edges}")
print(f"Node features shape: {cora_graph.x.shape}")
print(f"Edge index shape: {cora_graph.edge_index.shape}")

# CoraLuDataset stores node-level labels under 'node_labels' (not graph-level 'y').
# Assign to y for convenience in training.
if cora_graph.y is None and hasattr(cora_graph, 'node_labels') and cora_graph.node_labels is not None:
    cora_graph.y = cora_graph.node_labels

In [ ]:
# Class label utilities
class_names = [
    "Genetic_Algorithms",
    "Reinforcement_Learning",
    "Theory",
    "Rule_Learning",
    "Case_Based",
    "Probabilistic_Methods",
    "Neural_Networks",
]

def get_label_color(label):
    """Return a color for a class label."""
    return plt.get_cmap('Set1')(label / 7)

def get_label_name(label):
    """Return the class name for a label index."""
    return class_names[label]

# Get node labels (one-hot -> integer)
# The y attribute may be one-hot encoded or integer labels
if cora_graph.y.dim() > 1 and cora_graph.y.shape[-1] == 7:
    labels = cora_graph.y.numpy()
    labels_int = np.argmax(labels, axis=-1)
else:
    labels_int = cora_graph.y.numpy()
    labels = np.eye(7)[labels_int]

print(f"Label distribution:")
for i, name in enumerate(class_names):
    count = (labels_int == i).sum()
    print(f"  {name}: {count}")

## 2. Train/Test Split with Masks

For semi-supervised node classification on a single graph, we use
node masks to separate train and validation nodes. The model still
sees the full graph structure during training, but the loss is computed
only on the masked nodes.

In [ ]:
n_nodes = cora_graph.num_nodes
inds = np.arange(n_nodes)
ind_train, ind_val = train_test_split(inds, test_size=0.10, random_state=0)

train_mask = torch.zeros(n_nodes, dtype=torch.bool)
val_mask = torch.zeros(n_nodes, dtype=torch.bool)
train_mask[ind_train] = True
val_mask[ind_val] = True

print(f"Train nodes: {train_mask.sum().item()}, Val nodes: {val_mask.sum().item()}")

## 3. Train GCN for Node Classification

In [ ]:
node_feature_dim = cora_graph.x.shape[-1]
print(f"Node feature dimension: {node_feature_dim}")

model = GCNModel(
    node_dim=124,
    depth=3,
    gcn_units=124,
    gcn_activation="relu",
    gcn_pooling="sum",
    node_pooling="sum",
    output_units=[64, 16],
    output_activation="relu",
    output_final_activation="softmax",
    output_use_bias=[True, True, False],
    num_targets=7,
    output_embedding="node",       # Node-level output (not graph-level)
    use_node_embedding=False,       # Cora uses float features, not integer IDs
    node_input_dim=node_feature_dim,
)
model = model.to(device)
print(model)

In [ ]:
# Prepare the single graph as a batch
# For a single graph we create a Batch with batch_size=1
cora_batch = Batch.from_data_list([cora_graph]).to(device)
labels_tensor = torch.tensor(labels, dtype=torch.float32).to(device)

# Training loop for node classification
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

n_epochs = 300
epo_step = 10

train_accs = []
val_accs = []
train_losses = []
train_mask_dev = train_mask.to(device)
val_mask_dev = val_mask.to(device)

# Learning rate schedule
from kgcnn_torch.training.scheduler import LinearLearningRateScheduler
scheduler = LinearLearningRateScheduler(
    optimizer, learning_rate_start=1e-3, learning_rate_stop=1e-4,
    epo_min=260, epo=300
)

start = time.process_time()
for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()

    out = model(cora_batch)  # (N, 7)

    # Loss only on train nodes
    loss = loss_fn(out[train_mask_dev], labels_tensor[train_mask_dev])
    loss.backward()
    optimizer.step()
    scheduler.step()

    # Track accuracy
    with torch.no_grad():
        pred_labels = out.argmax(dim=-1).cpu().numpy()
        train_acc = (pred_labels[train_mask.numpy()] == labels_int[train_mask.numpy()]).mean()
        train_accs.append(train_acc)
        train_losses.append(loss.item())

    if (epoch + 1) % epo_step == 0:
        model.eval()
        with torch.no_grad():
            out = model(cora_batch)
            pred_labels = out.argmax(dim=-1).cpu().numpy()
            val_acc = (pred_labels[val_mask.numpy()] == labels_int[val_mask.numpy()]).mean()
            val_accs.append(val_acc)

stop = time.process_time()
print(f"Training time: {stop - start:.1f}s")
print(f"Final train accuracy: {train_accs[-1]:.4f}")
print(f"Final val accuracy:   {val_accs[-1]:.4f}")

In [ ]:
# Plot training curve
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, len(train_accs) + 1), train_accs, label='Train Accuracy', color='blue')
plt.plot(np.arange(epo_step, n_epochs + epo_step, epo_step), val_accs, label='Val Accuracy', color='red')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('GCN Node Classification on Cora')
plt.legend()
plt.show()

## 4. Implement GNNInterface for Node-Level Explanations

For node classification, the key difference from graph classification is:
- `predict()` returns the prediction for a **single node** (indexed by `node_index`).
- `masked_predict()` applies masks to the full graph but returns the prediction for the target node.
- `get_explanation()` builds a subgraph around the target node showing relevant edges.

In [ ]:
class ExplainableGCNNode(GNNInterface):
    """Wraps a GCN for node-level explanations with GNNExplainer."""

    def __init__(self, gnn_model, node_index):
        super().__init__()
        self.gnn_model = gnn_model
        self.gnn_model.eval()
        self.node_index = node_index

    def predict(self, data, **kwargs):
        """Get the prediction for the target node."""
        with torch.no_grad():
            out = self.gnn_model(data)  # (N, 7)
            return out[self.node_index:self.node_index+1]  # (1, 7)

    def masked_predict(self, data, edge_mask, feature_mask, node_mask, **kwargs):
        """Get prediction for the target node with masks applied."""
        from torch_geometric.data import Data, Batch

        x = data.x.clone()
        edge_attr = data.edge_attr.clone() if data.edge_attr is not None else None

        # Apply masks
        x = x * feature_mask.T.float()  # Feature mask: (F, 1) -> broadcast to (N, F)
        # Note: we do NOT apply node_mask for node classification
        # since masking out nodes would destroy the graph structure.
        if edge_attr is not None:
            edge_attr = edge_attr * edge_mask.float()

        masked_data = Data(
            x=x,
            edge_index=data.edge_index,
            edge_attr=edge_attr,
            batch=data.batch if hasattr(data, 'batch') and data.batch is not None else torch.zeros(x.size(0), dtype=torch.long, device=x.device),
        )
        if hasattr(data, 'edge_weight'):
            masked_data.edge_weight = data.edge_weight

        out = self.gnn_model(masked_data)
        return out[self.node_index:self.node_index+1]

    def get_number_of_nodes(self, data):
        return data.x.shape[0]

    def get_number_of_node_features(self, data):
        return data.x.shape[-1]

    def get_number_of_edges(self, data):
        return data.edge_index.shape[1]

    def get_explanation(self, data, edge_mask, feature_mask, node_mask, node_labels=None, **kwargs):
        """Build a NetworkX graph with relevance annotations."""
        edge_relevance = edge_mask[:, 0].detach().cpu().numpy()
        node_relevance = node_mask[:, 0].detach().cpu().numpy()
        features = data.x.detach().cpu().numpy()
        edges = data.edge_index.T.detach().cpu().numpy()

        g = nx.Graph()
        for i in range(features.shape[0]):
            label = int(node_labels[i]) if node_labels is not None else -1
            g.add_node(i, features=features[i], relevance=float(node_relevance[i]), label=label)
        for i in range(edges.shape[0]):
            e = edges[i]
            g.add_edge(int(e[0]), int(e[1]), relevance=float(edge_relevance[i]))
        return g

    def present_explanation(self, explanation, threshold=0.5):
        """Visualize the explanation subgraph around the target node."""
        # Filter to relevant edges
        relevant_edges = []
        edge_color_map = []
        for (v, u, rel) in explanation.edges.data('relevance'):
            if rel is not None and rel > threshold:
                relevant_edges.append((v, u))
                edge_color_map.append((0, 0, 0, rel))

        if not relevant_edges:
            print("No edges above threshold. Try lowering the threshold.")
            return

        subgraph = explanation.edge_subgraph(relevant_edges)
        node_colors = []
        for n in subgraph.nodes():
            label = subgraph.nodes[n].get('label', -1)
            rel = subgraph.nodes[n].get('relevance', 0.5)
            if label >= 0:
                r, g, b, a = get_label_color(label)
                node_colors.append((r, g, b, rel))
            else:
                node_colors.append((0.5, 0.5, 0.5, rel))

        nx.draw(subgraph, node_color=node_colors, edge_color=edge_color_map, with_labels=True)

## 5. Explain a Node Prediction

In [ ]:
# Choose a validation node to explain
val_indices = np.argwhere(val_mask.numpy() == 1)[:, 0]
node_index = int(val_indices[0])
print(f"Explaining node {node_index}")

# Create the node-level explainable wrapper
explainable_gcn_node = ExplainableGCNNode(model, node_index)

# Get the prediction for this node
prediction = explainable_gcn_node.predict(cora_batch)
predicted_label = prediction.argmax(dim=-1).item()
true_label = labels_int[node_index]

print(f"Predicted class: {predicted_label} ({get_label_name(predicted_label)})")
print(f"True class:      {true_label} ({get_label_name(true_label)})")

In [ ]:
# Setup GNNExplainer for node classification
explainer = GNNExplainer(
    explainable_gcn_node,
    optimizer_options={
        'edge_mask_loss_weight': 0.001,
        'edge_mask_norm_ord': 2,
        'feature_mask_loss_weight': 0,
        'feature_mask_norm_ord': 2,
        'node_mask_loss_weight': 0,
        'node_mask_norm_ord': 1,
    },
    lr=1.0,
    epochs=80,
    loss_fn='cross_entropy',
)

# Run the explainer
inspection_result = explainer.explain(
    cora_batch,
    inspection=True,
    verbose=True,
    device=device,
)

In [ ]:
# Visualize the explanation
plt.figure(figsize=(10, 8))
explanation = explainer.get_explanation(node_labels=labels_int)
explainer.present_explanation(explanation, threshold=0.1)
plt.title(f"GNNExplainer: Node {node_index} - {get_label_name(predicted_label)}")
plt.show()

## 6. Inspect the Optimization Process

In [ ]:
# Plot prediction probabilities over optimization
plt.figure(figsize=(8, 5))
preds_array = np.array(inspection_result['predictions'])
for i in range(7):
    plt.plot(preds_array[:, 0, i], color=get_label_color(i), label=get_label_name(i))
plt.xlabel('Iterations')
plt.ylabel('Class Probability')
plt.title('Node Prediction During Mask Optimization')
plt.legend(fontsize='small')
plt.show()

In [ ]:
# Plot loss curve
plt.figure(figsize=(8, 4))
plt.plot(inspection_result['total_loss'], color='black')
plt.xlabel('Iterations')
plt.ylabel('Total Loss')
plt.title('GNNExplainer Total Loss')
plt.show()

## 7. Compare with k-hop Neighborhood

For reference, visualize the 2-hop neighborhood around the target node.
The GNNExplainer should identify the most important subset of this neighborhood.

In [ ]:
# Build the full Cora graph in NetworkX
cora_nx = nx.Graph()
edges_np = cora_graph.edge_index.T.numpy()
cora_nx.add_nodes_from([(i, {"label": int(labels_int[i])}) for i in range(n_nodes)])
cora_nx.add_edges_from(edges_np.tolist())

# Extract 2-hop ego graph
khop_graph = nx.generators.ego.ego_graph(cora_nx, node_index, radius=2)

# Set uniform relevance for comparison
for n in khop_graph.nodes:
    khop_graph.nodes[n]['relevance'] = 1.0
for (u, v) in khop_graph.edges:
    khop_graph.edges[u, v]['relevance'] = 1.0

print(f"2-hop neighborhood of node {node_index}: "
      f"{khop_graph.number_of_nodes()} nodes, {khop_graph.number_of_edges()} edges")

# Visualize the 2-hop neighborhood
plt.figure(figsize=(10, 8))
node_colors = []
for n in khop_graph.nodes():
    label = khop_graph.nodes[n]['label']
    r, g, b, a = get_label_color(label)
    node_colors.append((r, g, b, 1.0))

nx.draw(khop_graph, node_color=node_colors, with_labels=True, font_size=8,
        node_size=200, edge_color='gray', alpha=0.8)
plt.title(f"2-hop Neighborhood of Node {node_index} (for comparison)")
plt.show()

## 8. Explain Multiple Nodes

In [ ]:
# Explain several validation nodes from different classes
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

# Pick nodes from different classes
nodes_to_explain = []
for cls in range(min(6, 7)):
    cls_val_nodes = val_indices[labels_int[val_indices] == cls]
    if len(cls_val_nodes) > 0:
        nodes_to_explain.append(int(cls_val_nodes[0]))

# Fill up if not enough classes in val set
while len(nodes_to_explain) < 6 and len(val_indices) > len(nodes_to_explain):
    candidate = int(val_indices[len(nodes_to_explain)])
    if candidate not in nodes_to_explain:
        nodes_to_explain.append(candidate)

for ax_idx, ni in enumerate(nodes_to_explain[:6]):
    # Create explainable wrapper for this node
    expl_gcn = ExplainableGCNNode(model, ni)

    # Get prediction
    pred = expl_gcn.predict(cora_batch)
    pred_label = pred.argmax(dim=-1).item()
    true_label_i = labels_int[ni]

    # Run explainer
    expl = GNNExplainer(
        expl_gcn,
        optimizer_options={
            'edge_mask_loss_weight': 0.001, 'edge_mask_norm_ord': 2,
            'feature_mask_loss_weight': 0, 'feature_mask_norm_ord': 2,
            'node_mask_loss_weight': 0, 'node_mask_norm_ord': 1,
        },
        lr=1.0, epochs=80, loss_fn='cross_entropy',
    )
    expl.explain(cora_batch, device=device)
    explanation = expl.get_explanation(node_labels=labels_int)

    # Visualize
    plt.sca(axes[ax_idx])
    relevant_edges = [(u, v) for (u, v, r) in explanation.edges.data('relevance')
                      if r is not None and r > 0.1]
    if relevant_edges:
        subg = explanation.edge_subgraph(relevant_edges)
        nc = []
        for n in subg.nodes():
            lbl = subg.nodes[n].get('label', -1)
            rel = subg.nodes[n].get('relevance', 0.5)
            if lbl >= 0:
                r, g, b, a = get_label_color(lbl)
                nc.append((r, g, b, rel))
            else:
                nc.append((0.5, 0.5, 0.5, rel))
        ec = [(0, 0, 0, explanation.edges[u, v].get('relevance', 0.5))
              for (u, v) in subg.edges()]
        nx.draw(subg, ax=axes[ax_idx], node_color=nc, edge_color=ec,
                with_labels=True, font_size=6, node_size=150)
    correct = 'OK' if pred_label == true_label_i else 'WRONG'
    axes[ax_idx].set_title(
        f"Node {ni}: true={get_label_name(true_label_i)}\n"
        f"pred={get_label_name(pred_label)} [{correct}]", fontsize=9)

plt.tight_layout()
plt.suptitle('GNNExplainer Node Explanations on Cora', fontsize=14, y=1.02)
plt.show()

## Summary

Node-level GNNExplainer in kgcnn-torch:

```python
# 1. Wrap the trained model with node-level GNNInterface
class ExplainableGCNNode(GNNInterface):
    def __init__(self, gnn_model, node_index):
        self.node_index = node_index
        ...

    def predict(self, data):
        return self.gnn_model(data)[self.node_index:self.node_index+1]

    def masked_predict(self, data, edge_mask, feature_mask, node_mask):
        # Apply masks, return prediction for target node only
        ...

# 2. Create explainer and explain
explainer = GNNExplainer(ExplainableGCNNode(model, node_idx), ...)
explainer.explain(graph_data)

# 3. Visualize subgraph explanation
explanation = explainer.get_explanation(node_labels=labels)
explainer.present_explanation(explanation, threshold=0.1)
```

Key differences from graph-level explanation:
- `predict()` and `masked_predict()` return the output for a **single node**.
- The explanation shows which neighboring nodes and edges most influence the target node's classification.